# Compression, Materialized Views, and Inverted Indexes

**Module 6: Table Properties, Materialized Views, and Indexes** · [Lesson](course.md) · [Quiz](quiz06_properties_mv_and_indexes.ipynb)

In Module 3 you found the failed checkouts with a search of the access log, and in Module 4 you kept the requests and failures of each service and minute in a table of your own. In this Lab you look at what makes such work fast and cheap in Doris: the properties of the exporter's `otel_logs`, the inverted index that answers the access-log search, a materialized view that Doris keeps up to date with the counts of Module 4, and the compression that decides how much disk the telemetry takes. Every cell can be rerun.

Plan about 40 minutes. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1).

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Check the failed checkouts in the access log

The search of this Lab is the one from Module 3: the storefront proxy's access-log lines of the failed checkouts, `"POST /api/checkout ... HTTP/1.1" 500`. Count them:

In [ ]:
failed_checkouts = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS failed_checkouts, MAX(timestamp) AS latest
    FROM otel_logs
    WHERE body MATCH_PHRASE 'POST /api/checkout'
      AND body MATCH_PHRASE 'HTTP/1.1 500'
    HAVING COUNT(*) > 0
    """,
    title="Access-log lines of failed checkouts in otel_logs",
    hint="No checkout has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

**Expected result**

One row: one access-log line per failed order since you turned on the flag, and the time of the latest in UTC.

## 2. Read the table's definition

`SHOW CREATE TABLE` returns the statement that would create a table as it is now, with all of its properties. `lab.show_create_table(...)` shows it as text:

In [ ]:
lab.show_create_table("otel_logs");

**Expected result**

The statement for `otel_logs`, about 60 lines. Read it from the top:

- **Columns.** `body` is a `text` column. `resource_attributes` and `log_attributes` are `variant`: Doris stores each key that the attributes contain as a column of its own, up to `variant_max_subcolumns_count` = 2048 keys, so a query on one key reads only that key.
- **Indexes.** One inverted index per column, `USING INVERTED`. Only `idx_body` has properties: `"parser" = "unicode"` splits each message into words, `"lower_case" = "true"` indexes them in lowercase, and `"support_phrase" = "true"` also records where each word occurs, which phrase searches need. Section 3 measures it.
- **Key, partitions, buckets.** `DUPLICATE KEY(timestamp, service_name)`, daily partitions, and `RANDOM BUCKETS AUTO`, which Modules 4 and 5 covered.
- **`compaction_policy` = `time_series`** and the `time_series_compaction_*` settings. Every load writes a small set of files to each tablet that it touches, a rowset, and compaction merges them in the background. The default policy merges small rowsets into larger ones in levels, so some data is rewritten many times. The time-series policy suits data that is only appended in time order: it waits until the rowsets of a tablet reach about `time_series_compaction_goal_size_mbytes` (1024 MB), or a number of files, or an hour, and merges them once into a final rowset. This saves disk and CPU on tables that receive many loads, such as these.
- **`group_commit_interval_ms` and `group_commit_data_bytes`.** They apply only to loads that use group commit, which collects many small loads into one transaction, here for 10 seconds or 128 MB. A load asks for it with a header, or the table makes it the default with `group_commit_mode`, which the definition would show. The exporter does neither; it sends each batch as one Stream Load, which its `batch` processor keeps large.
- **No `compression`.** The exporter created the table with `"compression" = "zstd"`. `SHOW CREATE TABLE` leaves out properties that are at their default value, and ZSTD is the default in Doris 4.x. Section 5 compares it with LZ4.

## 3. Measure what the index of `body` does for the search

In Module 3, Discover turned the search `body:"POST /api/checkout" AND body:"HTTP/1.1 500"` into two `MATCH_PHRASE` conditions. With `idx_body`, Doris looks up the words of each phrase in the index and reads only the rows that contain them in order. How much reading does that save? A Query Profile shows it. For each operator of a query, such as the scan of a table, Doris records what the operator did, including how many rows the scan skipped and why. Doris collects a profile for every query of a session in which the variable `enable_profile` is `true`. The next cell sets it for this notebook's session, together with three related variables:

- `profile_level = 2`: the profile includes the row counters of the scan. This is the default in Doris 4.1; level 1 leaves most of them out.
- `enable_sql_cache = false`: when a table has not changed for a while, the SQL cache can return the result of a repeated query without scanning the table, and the profile of that query shows no scan.
- `enable_condition_cache = false`: the condition cache remembers which parts of the data had no matching rows in an earlier run of a filter, and a rerun skips them, so the counters of the rerun no longer show all the rows that the index filters.

In [ ]:
lab.execute("""
SET enable_profile = true,
    profile_level = 2,
    enable_sql_cache = false,
    enable_condition_cache = false
""", title="Collect Query Profiles in this session, and scan the table on every run");

**Expected result**

A card says that the statement succeeded. The variables apply to this notebook's connection to Doris until the kernel restarts; after a restart, run the first cell of this Lab and this cell again. In a MySQL client, run the same `SET` statement before your queries.

Count the failed checkouts by service, and fetch the profile of the query. `lab.profile_query(...)` runs the query, then finds and fetches its profile in three steps, which you can also take in a MySQL client and a terminal:

1. `SELECT last_query_id()` returns the ID of the last query in the session. The ID of a query is also the ID of its profile.
2. `SHOW QUERY PROFILE` lists the newest profiles, each with its ID and SQL. When the query's ID is in the list, Doris has collected its profile.
3. The FE HTTP API returns the text of a profile by its ID. The cell shows the request as a `curl` command.

The FE web UI lists the same profiles: open http://127.0.0.1:28030 (port `DORIS_OBS_HTTP_PORT`), log in as `root` with no password, and open QueryProfile.

The query groups by service instead of only counting: for a `COUNT(*)` alone, Doris can answer from the index without reading any row (`COUNT_ON_INDEX` in the plan), and the counters would no longer show the rows that it reads.

In [ ]:
INDEX_QUERY = """
SELECT service_name, COUNT(*) AS failed_checkouts
FROM otel_logs
WHERE body MATCH_PHRASE 'POST /api/checkout'
  AND body MATCH_PHRASE 'HTTP/1.1 500'
GROUP BY service_name
"""
profile = lab.profile_query(INDEX_QUERY, title="Failed checkouts in the access log, by service")

**Expected result**

The result first: one row, `frontend-proxy`, with as many failed checkouts as the first cell found, or a few more.

Then the three steps: the query ID, the query's row in `SHOW QUERY PROFILE` with its start time and duration (`Total`, a few milliseconds), and the `curl` command, followed by the profile text in a collapsed box, a few thousand lines long. The text starts with a `Summary` of the query, and `ChangedSessionVariables` lists the variables that the `SET` cell changed from their defaults. Then the operators of the query plan follow twice. Doris runs each operator as several instances in parallel: `MergedProfile` combines the instances of each operator, and `DetailProfile` lists every instance on its own. The row counters of the table scan are in `DetailProfile`, in each `OLAP_SCAN_OPERATOR` under `SegmentIterator`: search the text for `RowsInvertedIndexFiltered`.

Each row of the table that the scan considers is counted once, by the step that removed it, or as a row that passed: skipped by the zone maps, skipped by the inverted index, read and dropped by a filter, or read and kept. Add up these counters over the scan instances:

In [ ]:
with_index = lab.scan_filters(profile, title="How the scan of otel_logs filtered its rows")
assert with_index["RowsInvertedIndexFiltered"] > 0, (
    "The inverted index filtered no rows. Check that the first cell of this Lab found failed checkouts, "
    "then rerun the query and the cells after it."
)

**Expected result**

Five counters, which add up to the number of rows that `otel_logs` had when the query ran:

- `RowsStatsFiltered`: 0, or a few rows. Doris stores the rows of a table in segments (data files), and keeps the minimum and maximum of each column for every segment and for every page of rows in it, the zone maps. They can rule out a range of values, not words inside a text, so they skip nothing for this search.
- `RowsInvertedIndexFiltered`: nearly every row of the table, tens of thousands. The index lists the rows that contain the words of both phrases in order, and Doris leaves out all other rows before it reads any data.
- `RowsVectorPredFiltered` and `RowsExprPredFiltered`: 0. These would be rows that Doris reads and then drops, after it compares them with the filter; with the index, Doris reads only matching rows.
- `RowsRead`: the failed checkouts, as many as the query counted.

Now measure the same query without the index. With `enable_inverted_index_query = false`, Doris evaluates filters without inverted indexes: it reads each row's `body`, splits it into words, and compares them with the phrases. The next cell turns the index off for this session, runs the query and adds up the counters of its profile, and turns the index on again:

In [ ]:
import pandas as pd

lab.execute("SET enable_inverted_index_query = false", title="Stop using inverted indexes in this session")
try:
    profile_without_index = lab.profile_query(INDEX_QUERY, title="Failed checkouts by service, without the index")
    without_index = lab.scan_filters(profile_without_index, title="How the scan filtered the rows without the index")
finally:
    # Use the index again, also when a step above failed.
    lab.execute("SET enable_inverted_index_query = true", title="Use inverted indexes again")

pd.DataFrame({"with the index": with_index, "without the index": without_index})

**Expected result**

The query returns the same count as before, plus the failures that arrived in between. The last table puts the two measurements side by side:

- `RowsInvertedIndexFiltered` is 0, and nearly as many rows appear under `RowsExprPredFiltered` instead. `MATCH_PHRASE` is a filter expression: Doris read the `body` of each of these rows, tokenized it, and found that it did not match.
- `RowsRead` is the same: the same rows match.

Without the index, Doris read every message of the table to find the few failed checkouts; with it, only those. In a table this small, both queries take milliseconds; in a table of billions of log lines, reading and tokenizing every message costs disk reads and CPU time on every search. A `LIKE '%...%'` filter reads every message the same way, which is why searches use `MATCH_ANY`, `MATCH_ALL`, and `MATCH_PHRASE` on a column with an index.

`RowsInvertedIndexFiltered` is evidence of what the index did in this scan of this query: it removed these rows before Doris read them. On its own, it does not show how much faster the query became, what else the query spent its time on, or whether another filter can use an index. Measure each query that you want to make such a claim about.

## 4. A materialized view of the incident's counts

In Module 4, you loaded the requests and failed requests of each service and minute into `requests_per_minute` with `INSERT` statements, and saw that a load job must load every span exactly once. A **materialized view** keeps such a summary without a job. A synchronous materialized view belongs to one table: when you create it, Doris computes it from the table's rows, and from then on it updates it in the same transaction as every load into the table. Queries keep naming the table, and Doris reads the view when the view can answer them.

Create a view on `otel_traces` with the counts of Module 4. Doris 4.1 rejects a view whose columns have the same names as the table's columns, so each column gets a name of its own, starting with `mv_`:

In [ ]:
created_after = lab.sql("SELECT NOW() AS doris_time", title="The time on Doris before the view is created")["doris_time"][0]
lab.execute("DROP MATERIALIZED VIEW IF EXISTS server_minutes_mv ON otel_traces",
            title="Drop the view if an earlier run created it, so that the cell can be rerun")
lab.execute("""
CREATE MATERIALIZED VIEW server_minutes_mv AS
SELECT date_trunc(end_time, 'minute') AS mv_request_minute,
       service_name AS mv_service_name,
       COUNT(*) AS mv_requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS mv_failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
""", title="Create the materialized view server_minutes_mv on otel_traces")
lab.wait_for_rows(
    f"""
    SHOW ALTER TABLE MATERIALIZED VIEW FROM otel
    WHERE TableName = 'otel_traces' AND RollupIndexName = 'server_minutes_mv'
      AND State = 'FINISHED' AND CreateTime >= '{created_after}'
    """,
    title="The job that builds server_minutes_mv, once it has finished",
    hint="Doris has not finished building the view. Check its State and Msg with SHOW ALTER TABLE "
         "MATERIALIZED VIEW FROM otel, then rerun this cell.",
);

**Expected result**

The time on Doris, two cards that say that the statements succeeded, then one row of `SHOW ALTER TABLE MATERIALIZED VIEW` with the `State` `FINISHED`, usually within a few seconds. `CREATE MATERIALIZED VIEW` returns as soon as Doris has accepted it: a job then builds the view from the rows that the table already has, and the view answers queries only after the job has finished. Check the job before you rely on a new view. `SHOW ALTER TABLE MATERIALIZED VIEW` also lists the jobs of views created earlier, so the cell asks only for jobs created after the time it noted.

The cell drops the view first because Doris 4.1.3 does not skip an existing view, even with `IF NOT EXISTS`: it reports that the view already exists, or, when its column names are taken, `Duplicate column name`. A rerun therefore rebuilds the view.

The view is stored with the table, as one of its indexes. `DESC ... ALL` lists the columns of the table and of each of its views:

In [ ]:
lab.sql("DESC otel_traces ALL", title="The columns of otel_traces and of its materialized view",
        columns=["IndexName", "IndexKeysType", "Field", "Type", "Extra", "DefineExpr", "WhereClause"])
lab.sql("""
SELECT (SELECT COUNT(*) FROM otel_traces WHERE span_kind = 'SPAN_KIND_SERVER') AS server_spans,
       (SELECT COUNT(*) FROM otel_traces INDEX server_minutes_mv) AS rows_in_the_view
""", title="Server spans in otel_traces, and rows in the view");

**Expected result**

The first table lists the 19 columns of `otel_traces`, then the view: `IndexName` `server_minutes_mv`, `IndexKeysType` `AGG_KEYS`, and four columns. `mv_request_minute` and `mv_service_name` are its key, `mv_requests` and `mv_failed_requests` have `SUM` in `Extra`, and `DefineExpr` gives the expression of each, with the view's `WHERE` clause. The view is an AGGREGATE index of the table: it combines rows by `SUM`, as `requests_per_minute` did in Module 4, but Doris fills it.

The second table: tens of thousands of server spans or more, and a few hundred to a few thousand rows in the view, one per service and minute. `SELECT ... FROM otel_traces INDEX server_minutes_mv` reads the view directly; a query names the table, and Doris decides.

Ask `otel_traces` for the requests and failed requests of each service and minute, and look at the plan:

In [ ]:
MINUTES_QUERY = """
SELECT date_trunc(end_time, 'minute') AS request_minute,
       service_name,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
"""
lab.explain_scan(MINUTES_QUERY, title="The plan of the counts per service and minute");

**Expected result**

One scan of `otel.otel_traces` whose `reads` column is `server_minutes_mv`: Doris rewrote the query to read the view's few hundred rows instead of the table's spans. The second table lists the view as `chosen`. The result of the query is the same either way; run `lab.sql(MINUTES_QUERY)` to see it.

Doris uses a view only when it can compute the query's result from the view's rows. Look at the plans of three queries that differ from the view in one way each: a filter on `end_time` itself, no filter on `span_kind`, and the maximum duration, which the view does not keep.

In [ ]:
for title, query in (
    ("A filter on end_time", """
SELECT date_trunc(end_time, 'minute') AS request_minute, COUNT(*) AS requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER' AND service_name = 'payment'
  AND end_time >= UTC_TIMESTAMP() - INTERVAL 30 MINUTE
GROUP BY date_trunc(end_time, 'minute')
"""),
    ("Every span, not only server spans", """
SELECT date_trunc(end_time, 'minute') AS request_minute, service_name, COUNT(*) AS spans
FROM otel_traces
GROUP BY date_trunc(end_time, 'minute'), service_name
"""),
    ("The maximum duration", """
SELECT date_trunc(end_time, 'minute') AS request_minute, service_name, MAX(duration) AS max_duration
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
GROUP BY date_trunc(end_time, 'minute'), service_name
"""),
):
    lab.explain_scan(query, title=f"{title}: the plan");

**Expected result**

All three plans read `otel_traces` itself, and list the view as `not usable`, each with a reason:

- **A filter on `end_time`**: `Rewrite compensate predicate by view fail`. The view keeps only the minute of each span, not its `end_time`, so Doris cannot apply the filter to the view's rows. A filter on the view's expression, such as `date_trunc(end_time, 'minute') >= ...`, would let it use the view; the exercise does that.
- **Every span**: `Predicate compensate fail`. The view holds only server spans; it cannot count the others.
- **The maximum duration**: `Query function roll up fail`. The view keeps counts and sums, not durations.

The rewrite is structural: Doris matches the query's expressions, filters, and aggregates against the view's, and does not try to prove that two differently written expressions mean the same. A view pays off for queries that dashboards and alerts run again and again in the same shape. It also has a cost: every load into `otel_traces` now updates the view as well, and the view takes disk.

## 5. Compare compression algorithms

Doris compresses each column of each page before it writes it to disk. The algorithm is a table property, `compression`, set when the table is created. ZSTD, the default, compresses more; LZ4 compresses less but decompresses with less CPU work. To compare them, everything else must be the same: the same rows, loaded the same way, into tables that differ only in `compression`. The next cell creates two such tables, `logs_zstd` and `logs_lz4`, with the main columns of `otel_logs` and no indexes, and loads the same log rows into each with one `INSERT`:

In [ ]:
for algorithm in ("zstd", "lz4"):
    lab.execute(f"""
CREATE TABLE IF NOT EXISTS logs_{algorithm} (
    timestamp           DATETIME(6),
    service_name        VARCHAR(200),
    trace_id            VARCHAR(200),
    span_id             STRING,
    severity_number     INT,
    severity_text       STRING,
    body                STRING,
    resource_attributes VARIANT,
    log_attributes      VARIANT
)
DUPLICATE KEY(timestamp, service_name)
DISTRIBUTED BY RANDOM BUCKETS 1
PROPERTIES ("compression" = "{algorithm}")
""", title=f"Create logs_{algorithm}")

copy_limit = lab.sql("SELECT MAX(timestamp) - INTERVAL 30 SECOND AS copy_limit FROM otel_logs",
                     title="Copy the log rows before this time")["copy_limit"][0]
for algorithm in ("zstd", "lz4"):
    lab.execute(f"TRUNCATE TABLE logs_{algorithm}", title="Empty the table, so that the cell can be rerun")
    lab.execute(f"""
INSERT INTO logs_{algorithm}
SELECT timestamp, service_name, trace_id, span_id, severity_number, severity_text, body, resource_attributes, log_attributes
FROM otel_logs
WHERE timestamp < '{copy_limit}'
""", title=f"Copy the same log rows into logs_{algorithm}")
copied = lab.sql("SELECT COUNT(*) AS copied_rows FROM logs_zstd", title="Rows copied")["copied_rows"][0]

**Expected result**

Two `CREATE TABLE` cards, the copy limit, and two inserts that report the same number of rows, tens of thousands. Each table has one bucket and received one load, so each holds its rows in one rowset, laid out the same way.

Doris counts a table's size from the reports of its BEs, which arrive every minute or so. The next cell waits until both tables report all of their rows, then shows their sizes:

In [ ]:
sizes = lab.wait_for_rows(
    f"""
    SELECT TABLE_NAME AS table_name,
           TABLE_ROWS AS table_rows,
           DATA_LENGTH AS data_bytes,
           ROUND(DATA_LENGTH / 1048576, 2) AS data_mb
    FROM information_schema.tables
    WHERE TABLE_SCHEMA = 'otel'
      AND TABLE_NAME IN ('logs_zstd', 'logs_lz4')
      AND TABLE_ROWS = {copied}
      AND DATA_LENGTH > 0
    ORDER BY data_bytes
    """,
    title="Sizes of the two tables, once the BEs have reported them",
    hint="The BEs have not reported the tables' sizes yet. Rerun this cell in a minute.",
    min_rows=2,
    timeout_seconds=180,
)
print(f"LZ4 needs {sizes['data_bytes'].max() / sizes['data_bytes'].min():.2f} times the space of ZSTD for the same rows.")

**Expected result**

Within one to two minutes, two rows with the same `table_rows`: `logs_zstd` first and smallest, `logs_lz4` larger, and the ratio, 1.5 to 1.6 for the course's logs: LZ4 needs 50% to 60% more disk for the same rows. The access log's lines repeat a lot from line to line, which ZSTD exploits better.

The comparison holds for these rows and this layout. Other data compresses differently, and the table's indexes and its compaction add to the size of a real table. It also says nothing about speed: LZ4 decompresses with less CPU work, which matters for tables that are read far more than they are stored, such as a small table of hot data. For telemetry, which is written once, kept for days, and mostly read in small parts, the exporter's choice of ZSTD saves disk. The algorithm is set when a table is created; to change it, create a new table and copy the data.

## 6. Exercise: a query that the view answers

Write a query on `otel_traces` that returns the payment service's requests and failed requests of each minute, for the last 30 minutes, and that Doris answers from `server_minutes_mv`. Check its plan with `lab.explain_scan(...)`: `reads` must be `server_minutes_mv`. Section 4 showed which filter on time stops Doris from using the view.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
PAYMENT_MINUTES = """
SELECT date_trunc(end_time, 'minute') AS request_minute,
       COUNT(*) AS requests,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER'
  AND service_name = 'payment'
  AND date_trunc(end_time, 'minute') >= date_trunc(UTC_TIMESTAMP() - INTERVAL 30 MINUTE, 'minute')
GROUP BY date_trunc(end_time, 'minute')
ORDER BY request_minute
"""
lab.sql(PAYMENT_MINUTES, title="The payment service's requests per minute")
lab.explain_scan(PAYMENT_MINUTES, title="The plan of the payment service's requests per minute");

**Expected result**

One row per minute in which the payment service handled a charge, up to 31, with one to four requests and about half of them failed. The plan reads `server_minutes_mv`, and its `predicates` name the view's columns, `mv_service_name` and `mv_request_minute`: the filter on the minute, not on `end_time`, lets Doris apply it to the view's rows.

## You are done when

- you can say what the time-series compaction policy, the group commit settings, and the missing `compression` in the definition of `otel_logs` mean;
- your measurement shows the inverted index of `body` removing nearly every row before Doris reads it, and the same search without it reading every message;
- `server_minutes_mv` is `FINISHED`, and the plan of the counts per service and minute reads it;
- you have measured the size of the same log rows with ZSTD and with LZ4.

## Clean up

The view adds work to every load of `otel_traces`, and the later modules do not use it or the two log tables. To remove them, run `DROP MATERIALIZED VIEW IF EXISTS server_minutes_mv ON otel_traces`, `DROP TABLE IF EXISTS logs_zstd`, and `DROP TABLE IF EXISTS logs_lz4` with `lab.execute(...)` or in a MySQL client. The session variables of section 3 last until the kernel restarts.

## Next

Open [Quiz 6: Table Properties, Materialized Views, and Indexes](quiz06_properties_mv_and_indexes.ipynb), then continue to [Module 7: Trace Analysis and Litefuse](../module07-data-analysis-litefuse/course.md).